# Phase V: the original vec2vec on our data, with a fixed time budget

Neural Nexus, vector-DB compression project. Two runs in total: `SCHEME = "fp32"` and `SCHEME = "pq128x8"`.

Why: our main attacker is mini-vec2vec (linear, minutes on a CPU). This notebook measures how long the
original vec2vec (a GAN, Jha et al. 2025) needs on OUR data, and how good it gets, so the paper can show
the comparison with real numbers instead of quoting "1 to 7 days".

What it does:
1. Downloads the authors' official vec2vec code (fixed commit 5efb3c0) and runs THEIR trainer with THEIR
   settings (configs/unsupervised.toml). Only the data input is changed: it reads our saved vectors.
2. Victim side: the 50,000 victim BGE-large vectors (32 tokens), uncompressed or PQ-decoded.
   Attacker side: the 200,000 A2 GTR vectors. No pairs are given to training.
3. After every epoch it scores the translation on 8,192 victim passages against their true GTR vectors
   (top-1, average rank, mean cosine). Same metric as mini-vec2vec, so the two are directly comparable.
4. Stops after 10.5 hours, or earlier if the authors' own early-stopping rule fires.
5. Saves the log, the latest and the best translator, a plot, and a summary.

Settings: Accelerator = GPU T4 (x1 is enough), Internet = On.
Inputs (Add Input -> Notebooks): `phaseA_fp32_embeddings`, `phaseA2b_verify_attacker_extra`,
and for `pq128x8` also the teammate's `phaseB-pq128x8` notebook.
Run with Save Version -> Save & Run All (Commit). Expected time: up to about 11 hours.

In [ ]:
# Cell 1 - install + official vec2vec code at a fixed commit
!pip install -q "transformers==4.46.3" "vec2text==0.0.13" "sentence-transformers==3.4.1" accelerate toml wandb evaluate diffusers nltk seaborn
!rm -rf /kaggle/working/vec2vec && git clone -q https://github.com/rjha18/vec2vec.git /kaggle/working/vec2vec && cd /kaggle/working/vec2vec && git checkout -q 5efb3c02ee908847581c6488cf9c8ed6dace3c29 && git log -1 --format='vec2vec commit %H %cd'

In [ ]:
# Cell 2 - settings. CHANGE ONLY SCHEME.
SCHEME = "fp32"          # "fp32" or "pq128x8"
BUDGET_HOURS = 10.5      # Kaggle stops a session at 12 h; keep ~1 h for setup and saving
SEED = 5                 # the authors' default seed in configs/unsupervised.toml
EVAL_N = 8192            # same evaluation size as the vec2vec and mini-vec2vec papers
L, MODEL = 32, "bge-large"

import os, glob, json, time
import numpy as np, torch
assert torch.cuda.is_available(), "Set Accelerator = GPU T4"
OUT = f"/kaggle/working/phaseV_{SCHEME}"
os.makedirs(OUT, exist_ok=True)

def find_one(pattern, what):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert len(hits) == 1, f"{what}: expected exactly one '{pattern}' in the inputs, found {hits}"
    return hits[0]
IN_A = os.path.dirname(find_one("manifest.json", "Phase A output"))
IN_A2 = os.path.dirname(find_one("manifest_A2.json", "Phase A2b output"))
assert json.load(open(f"{IN_A}/manifest.json"))["phase"] == "A_fp32_embeddings"
if SCHEME == "fp32":
    victim_path = f"{IN_A}/{MODEL}_victim_L{L}.npy"
else:
    victim_path = find_one(f"decoded_victim_{SCHEME}_{MODEL}_L{L}.npy", f"Phase B {SCHEME} output")
print("victim vectors:", victim_path)

In [ ]:
# Cell 3 - what we expect BEFORE the run (so the run cannot move the goalposts)
EXPECT = {
    "setup (install + clone + load)": "5-10 minutes",
    "steps per epoch": "50,000 / 256 = 195 (the smaller side sets the epoch length)",
    "minutes per epoch on a T4": "unknown before this run; rough guess 0.5-2",
    "random-guess top-1 (8,192 candidates)": "about 0.0001",
    "if training works (fp32)": "top-1 rises clearly above 0.01 within the first 1-2 hours and ends around 0.5-0.85. "
                                "The paper reports 0.82-0.99 with 1M vectors per side; its own data ablation gives ~0.81 with 50k on one side",
    "if the GAN fails for this seed": "top-1 stays near 0.0001 for hours. The paper itself says cross-backbone runs often fail "
                                      "(e5->gtr: 3 of 15 seeds converged in 10 epochs). That is a result, not a bug",
    "pq128x8 vs fp32": "equal or lower top-1 than fp32",
    "stop reason": "time budget (10.5 h) or the authors' early stopping (after epoch 80, no improvement for 20 epochs)",
}
for k, v in EXPECT.items(): print(f"- {k}: {v}")

In [ ]:
# Cell 4 - load data and check it
X_victim = np.load(victim_path).astype(np.float32)
Y_att = np.load(f"{IN_A2}/gtr_attextra_L{L}.npy").astype(np.float32)
Y_true = np.load(f"{IN_A}/gtr_victim_L{L}.npy").astype(np.float32)    # scoring only, never trained on
eval_idx = np.random.default_rng(0).choice(len(X_victim), EVAL_N, replace=False)
X_eval, Y_eval = X_victim[eval_idx], Y_true[eval_idx]
print("victim", X_victim.shape, "attacker", Y_att.shape, "eval", X_eval.shape, Y_eval.shape)
assert X_victim.shape == (50000, 1024) and Y_att.shape == (200000, 768) and Y_true.shape == (50000, 768)
assert np.isfinite(X_victim).all() and np.isfinite(Y_att).all()
if SCHEME == "fp32":
    assert abs(np.linalg.norm(X_victim, axis=1).mean() - 1) < 1e-3

In [ ]:
"""Run the OFFICIAL vec2vec trainer (github.com/rjha18/vec2vec, commit 5efb3c0) on our saved vectors.

What is changed, and nothing else:
  1. Data: instead of embedding text on the fly, the trainer gets our saved vectors through its own
     "precomputed embeddings" path (the one the authors use for fMRI data). Victim side = BGE vectors
     (fp32 or decoded compressed), attacker side = GTR vectors. No pairs are given to training.
  2. Encoder sizes: that path assumes both spaces have the same size; ours are 1024 (BGE) and 768 (GTR).
  3. A wall-clock budget: training stops cleanly when the budget is used up.
  4. After every epoch we score the translation with OUR metric (same one as mini-vec2vec):
     top-1 / average rank / mean cosine of victim vectors translated into GTR space vs their true GTR vectors.
     The true GTR vectors are used only for this scoring, never for training.
  5. wandb logging is replaced by a local JSON-lines log.
All losses, architecture, optimiser and hyper-parameters come from the authors' configs/unsupervised.toml.
"""
import os, sys, json, time, types, inspect, shutil

def run(repo_dir, X_victim, Y_attacker, X_eval, Y_eval, out_dir, budget_hours,
        seed=5, val_size=4096, extra_args=None, log=print):
    import numpy as np, torch
    t_start = time.time()
    os.makedirs(out_dir, exist_ok=True)
    sys.path.insert(0, repo_dir)
    os.chdir(repo_dir)

    # beir is only used by the text-dataset path we do not use; stub it so it need not be installed
    if "beir" not in sys.modules:
        for name in ["beir", "beir.datasets", "beir.datasets.data_loader_hf"]:
            sys.modules[name] = types.ModuleType(name)
        sys.modules["beir.datasets.data_loader_hf"].HFDataLoader = object

    # Windows: vec2text imports the Unix-only "resource" module; give it a harmless stand-in
    if os.name == "nt" and "resource" not in sys.modules:
        try:
            import resource  # noqa: F401  (already provided, e.g. by a shim)
        except ImportError:
            stub = types.ModuleType("resource")
            stub.RLIMIT_NOFILE = 7
            stub.getrlimit = lambda *_: (8192, 8192)
            stub.setrlimit = lambda *_: None
            sys.modules["resource"] = stub

    import wandb
    import train as T
    from fmri.data import FmriSingleViewDataset

    # Windows: data-loader worker processes use "spawn" and would have to copy the whole dataset;
    # their helper also uses a Linux-only call. Load data in the main process instead (no speed
    # loss here: the data are already tensors in memory).
    if os.name == "nt":
        T.get_num_proc = lambda: 0

    class PairedDifferentSizes(torch.utils.data.Dataset):
        """Same as the authors' FmriPairedDataset, but the two spaces may have different sizes."""
        def __init__(self, name_a, Za, name_b, Zb):
            assert len(Za) == len(Zb)
            self.a, self.Za, self.b, self.Zb = name_a, Za, name_b, Zb
        def __len__(self):
            return len(self.Za)
        def __getitem__(self, i):
            return {self.a: self.Za[i], self.b: self.Zb[i]}

    if not torch.cuda.is_available():          # their helper divides by the GPU count (CPU-only test runs)
        T.get_num_proc = lambda: 2

    # -- 5. local logging instead of wandb
    log_path = os.path.join(out_dir, "v2v_log.jsonl")
    def _wandb_log(d, *a, **k):
        rec = {"t_hours": (time.time() - t_start) / 3600}
        for key, v in d.items():
            if isinstance(v, (int, float)):
                rec[key] = float(v)
        with open(log_path, "a") as f:
            f.write(json.dumps({"kind": "authors_val", **rec}) + "\n")
    wandb.log = _wandb_log
    wandb.Image = lambda *a, **k: None

    # -- 1. our data through the precomputed path
    Xv = torch.tensor(np.asarray(X_victim, dtype=np.float32))
    Ya = torch.tensor(np.asarray(Y_attacker, dtype=np.float32))
    Xe = torch.tensor(np.asarray(X_eval, dtype=np.float32))
    Ye = torch.tensor(np.asarray(Y_eval, dtype=np.float32))
    def _loader(*a, **k):
        supset = FmriSingleViewDataset("gtr", Ya)          # attacker's own space ("supervised" = known model)
        unsupset = FmriSingleViewDataset("bge", Xv)        # stolen victim vectors ("unsupervised" = unknown model)
        n = min(val_size, len(Xe))
        val = PairedDifferentSizes("gtr", Ye[:n], "bge", Xe[:n])
        return supset, unsupset, val, {"gtr": Ya.shape[1], "bge": Xv.shape[1]}
    T.load_fmri_train_and_eval_datasets = _loader

    # -- 4. our metric after every epoch
    Xe_n = torch.nn.functional.normalize(Xe, dim=1)
    Ye_n = torch.nn.functional.normalize(Ye, dim=1)
    best = {"top1": -1.0}
    state = {"epoch": 0}
    def _score(translator):
        dev = next(translator.parameters()).device
        was_training = translator.training
        translator.eval()
        outs = []
        with torch.no_grad():
            for i in range(0, len(Xe_n), 1024):
                ins = {"bge": Xe_n[i:i+1024].to(dev), "gtr": Ye_n[i:i+1024].to(dev)}
                _, tr = translator(ins, include_reps=False)
                outs.append(tr["gtr"]["bge"].float().cpu())
        if was_training:
            translator.train()
        Z = torch.nn.functional.normalize(torch.cat(outs), dim=1)
        S = Z @ Ye_n.T
        d = S.diag()
        ranks = (S > d[:, None]).sum(1) + 1
        return {"top1": float((ranks == 1).float().mean()), "avg_rank": float(ranks.float().mean()),
                "mean_cos": float(d.mean())}

    orig_loop = T.training_loop_
    def _loop(*a, **k):
        t0 = time.time()
        r = orig_loop(*a, **k)
        state["epoch"] += 1
        sc = _score(k["translator"])
        rec = {"kind": "ours", "epoch": state["epoch"], "t_hours": (time.time() - t_start) / 3600,
               "epoch_minutes": (time.time() - t0) / 60, **sc}
        with open(log_path, "a") as f:
            f.write(json.dumps(rec) + "\n")
        log(json.dumps(rec))
        # crash safety: keep the latest and the best translator outside the authors' save_dir
        model = k["accelerator"].unwrap_model(k["translator"])
        torch.save(model.state_dict(), os.path.join(out_dir, "translator_latest.pt"))
        if sc["top1"] > best["top1"]:
            best.update(sc, epoch=state["epoch"])
            torch.save(model.state_dict(), os.path.join(out_dir, "translator_best.pt"))
            with open(os.path.join(out_dir, "best.json"), "w") as f:
                json.dump(best, f)
        return r
    T.training_loop_ = _loop

    # -- 2 + 3. patch two lines of main(): per-space encoder sizes, and the time budget
    src = inspect.getsource(T.main)
    a = "encoder_dims = {cfg.sup_emb: int(embed_dim), cfg.unsup_emb: int(embed_dim)}"
    b = "encoder_dims = {cfg.sup_emb: int(embed_dim[cfg.sup_emb]), cfg.unsup_emb: int(embed_dim[cfg.unsup_emb])}"
    c = "    for epoch in range(max_num_epochs):\n"
    d = c + "        if _TIME_UP():\n            print('Time budget reached, stopping.')\n            break\n"
    e = "drop_last=cfg.train_drop_last if hasattr(cfg, 'train_drop_last') else not fmri_mode"
    f_ = "drop_last=bool(cfg.train_drop_last) if hasattr(cfg, 'train_drop_last') else not fmri_mode"
    g = "wandb.log(val_res)"   # wandb.init() re-binds wandb.log, so route the call explicitly
    assert src.count(a) == 1 and src.count(c) == 1 and src.count(e) == 2 and src.count(g) == 1, \
        "vec2vec code changed: patch points not found"
    src = src.replace(a, b).replace(c, d).replace(e, f_).replace(g, "_WANDB_LOG(val_res)")
    T.__dict__["_WANDB_LOG"] = _wandb_log
    # the budget counts training time from the start of the first epoch (setup time excluded),
    # and at least one epoch always runs
    clock = {"t0": None}
    def _time_up():
        if clock["t0"] is None:
            clock["t0"] = time.time()
            return False
        return (time.time() - clock["t0"]) / 3600 > budget_hours
    T.__dict__["_TIME_UP"] = _time_up
    exec(compile(src, T.__file__, "exec"), T.__dict__)

    args = {"unsup_emb": "bge", "sup_emb": "gtr", "dataset": "fmri_precomputed", "epochs": 100000,
            "use_wandb": 0, "seed": seed, "val_size": val_size, "save_dir": out_dir.rstrip("/") + "/authors_{}/",
            "force_wandb_name": 1, "wandb_name": "run",
            # our two sides have different sizes (50k vs 200k): drop the last partial batch so every
            # step pairs two full batches, as the text path of the authors' code does by default
            "train_drop_last": 1}
    args.update(extra_args or {})
    argv = ["train.py", "unsupervised"]
    for k_, v_ in args.items():
        argv += [f"--{k_}", str(v_)]
    sys.argv = argv
    T.argv = argv
    T.main()
    return best, log_path

In [ ]:
# Cell 6 - run (this is the long cell)
t0 = time.time()
best, log_path = run("/kaggle/working/vec2vec", X_victim, Y_att, X_eval, Y_eval, OUT,
                     budget_hours=BUDGET_HOURS, seed=SEED)
print(f"finished after {(time.time()-t0)/3600:.2f} h; best: {best}")

In [ ]:
# Cell 7 - results: table, plot, summary
import pandas as pd, matplotlib.pyplot as plt
os.chdir("/kaggle/working")
rows = [json.loads(l) for l in open(log_path)]
ours = pd.DataFrame([r for r in rows if r["kind"] == "ours"])
last = ours.iloc[-1].to_dict()
summary = {"scheme": SCHEME, "seed": SEED, "budget_hours": BUDGET_HOURS,
           "hours_used": float(ours["t_hours"].iloc[-1]), "epochs": int(ours["epoch"].iloc[-1]),
           "minutes_per_epoch_median": float(ours["epoch_minutes"].median()),
           "final": {k: float(last[k]) for k in ["top1", "avg_rank", "mean_cos"]},
           "best_by_true_pairs": best,
           "note": "final = honest unsupervised result; best = picked using the true pairs (optimistic upper bound)",
           "expected": EXPECT}
json.dump(summary, open(f"{OUT}/summary_{SCHEME}.json", "w"), indent=1)
print(json.dumps(summary, indent=1))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(ours["t_hours"], ours["top1"], color="tab:red", label="vec2vec top-1")
ax[0].axhline(1 / EVAL_N, color="gray", ls="--", lw=1, label="random guess")
ax[0].set_xlabel("Training time (hours, T4)"); ax[0].set_ylabel("Top-1 accuracy (8,192 candidates)")
ax[0].set_title(f"vec2vec on {SCHEME}: top-1 vs time"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(ours["t_hours"], ours["mean_cos"], color="tab:blue", label="vec2vec mean cosine")
ax[1].set_xlabel("Training time (hours, T4)"); ax[1].set_ylabel("Mean cosine to true GTR vector")
ax[1].set_title(f"vec2vec on {SCHEME}: cosine vs time"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT}/curve_{SCHEME}.png", dpi=150); plt.show()
shutil.rmtree("/kaggle/working/vec2vec", ignore_errors=True)   # do not keep the code copy in the output
print(sorted(os.listdir(OUT)))

## Reading the result

- `final.top1` is the honest number: what an attacker gets after this much training, with no access to pairs.
- `best_by_true_pairs` picks the best epoch using the real answers. A real attacker cannot do that, so it is an upper bound.
- `hours_used` and `epochs` are how long vec2vec needed (or got) on our data. Compare with mini-vec2vec in Phase C.
- If top-1 stayed near 0.0001 the whole time, the GAN did not converge for this seed. Report it as is; do not rerun silently.